# MR1a — Daily Crash MapReduce
**Input**: `crash_manhattan_alcohol_clean.csv`
**Output**: `daily_crash_mr.csv`

Schema: `zone_id, crash_date, crashes, injured, killed, avg_lat, avg_lon`

In [2]:
import h3
import pandas as pd
import numpy as np
from collections import defaultdict

INPUT  = 'crash_manhattan_alcohol_clean.csv'
OUTPUT = 'daily_crash_mr.csv'
H3_RES = 10
LAT_BOUNDS = (40.70, 40.88)
LON_BOUNDS = (-74.02, -73.91)

df = pd.read_csv(INPUT, dtype=str, low_memory=False)
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

df['latitude']  = pd.to_numeric(df['latitude'],  errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
df['crash_date'] = pd.to_datetime(df['crash_date'], errors='coerce').dt.strftime('%Y-%m-%d')

df = df[
    df['latitude'].between(*LAT_BOUNDS) &
    df['longitude'].between(*LON_BOUNDS)
].copy()

df['zone_id'] = df.apply(
    lambda r: h3.latlng_to_cell(r['latitude'], r['longitude'], H3_RES), axis=1
)

print(f'Input rows (clean, in-bounds): {len(df):,}')
print(f'Unique zones: {df["zone_id"].nunique():,}')

Input rows (clean, in-bounds): 2,887
Unique zones: 1,488


In [3]:
# ── MAP ───────────────────────────────────────────────────────────────────────
# key=(zone_id, crash_date), val=(crashes=1, injured, killed, lat, lon)

def mapper(row):
    key = (str(row['zone_id']), str(row['crash_date']))
    val = {
        'crashes': 1,
        'injured': float(pd.to_numeric(row.get('number_of_persons_injured', 0), errors='coerce') or 0),
        'killed':  float(pd.to_numeric(row.get('number_of_persons_killed',  0), errors='coerce') or 0),
        'lat':     row['latitude'],
        'lon':     row['longitude'],
    }
    return key, val

mapped = [mapper(row) for _, row in df.iterrows()]
print(f'Mapped pairs: {len(mapped):,}')

Mapped pairs: 2,887


In [ ]:
# ── REDUCE ────────────────────────────────────────────────────────────────────

acc = defaultdict(lambda: {'crashes':0,'injured':0.0,'killed':0.0,'lat':[],'lon':[]})

for key, val in mapped:
    a = acc[key]
    a['crashes'] += val['crashes']
    a['injured'] += val['injured']
    a['killed']  += val['killed']
    if pd.notna(val['lat']): a['lat'].append(float(val['lat']))
    if pd.notna(val['lon']): a['lon'].append(float(val['lon']))

rows = []
for (zone_id, crash_date), a in acc.items():
    rows.append({
        'zone_id':    zone_id,
        'crash_date': crash_date,
        'crashes':    a['crashes'],
        'injured':    a['injured'],
        'killed':     a['killed'],
        'avg_lat':    float(np.mean(a['lat'])) if a['lat'] else np.nan,
        'avg_lon':    float(np.mean(a['lon'])) if a['lon'] else np.nan,
    })

# ── OUTPUT ────────────────────────────────────────────────────────────────────
out = pd.DataFrame(rows).sort_values(['zone_id','crash_date']).reset_index(drop=True)
out.to_csv(OUTPUT, index=False)

print(f'Output rows : {len(out):,}')
print(f'Unique zones: {out["zone_id"].nunique():,}')
print(f'Saved → {OUTPUT}')
out.head()

Output rows : 2,871
Unique zones: 1,488
Saved → daily_crash_mr.csv


,zone_id,crash_date,crashes,injured,killed,avg_lat,avg_lon
0,8a2a1008800ffff,2017-02-11,1,2.0,0.0,40.796528,-73.974120
1,8a2a1008800ffff,2019-02-15,1,1.0,0.0,40.796307,-73.974434
2,8a2a1008802ffff,2020-11-05,1,0.0,0.0,40.797886,-73.973465
3,8a2a1008802ffff,2022-10-28,1,1.0,0.0,40.797150,-73.973660
4,8a2a10088047fff,2015-04-01,1,0.0,0.0,40.795252,-73.973209
